# EasyRAG QueryResult to answer

## Chapter position

This notebook is the map for the generation chapter. Retrieval has already done its part. The remaining job is to decide which citations survive, how they get packed, and how the final answer stays tied to those sources.

## Learning question

How does EasyRAG turn a structured `QueryResult` into a grounded `AnswerResult` without hiding the intermediate state?

## Success criteria

- You can point to the exact function that selects citations, packs context, builds the prompt, and synthesizes the answer.
- You can explain why `QueryResult.metadata` shows up again inside `AnswerResult.metadata`.
- You can tell the difference between raw retrieved citations and the smaller set the answer stage actually uses.

## Source code anchors

- `easyrag.rag.types.QueryResult`
- `easyrag.rag.generation.selection.select_answer_citations`
- `easyrag.rag.generation.packing.build_context_block`
- `easyrag.rag.generation.prompting.build_generation_prompt`
- `easyrag.rag.generation.pipeline.generate_answer`

## Method principles

- `easyrag.rag.types.QueryResult`: This is the retrieval handoff object. It keeps chunks, citations, relations, and metadata together so later generation or evaluation code can inspect evidence instead of starting from raw storage records.
- `easyrag.rag.generation.selection.select_answer_citations`: This is the answer-side evidence budgeter. It walks citations in order and keeps a compact subset that fits the item and character limits.
- `easyrag.rag.generation.packing.build_context_block`: This helper serializes selected citations into the numbered context block that the prompt builder reuses. It is the visible bridge between citation objects and prompt text.
- `easyrag.rag.generation.prompting.build_generation_prompt`: This prompt builder combines answer instructions with the packed evidence block. It is where style, citation requirements, and abstention rules become model-facing text.
- `easyrag.rag.generation.pipeline.generate_answer`: This is the answer orchestration wrapper. It reruns citation selection, packs context, builds the prompt, calls synthesis, and returns a structured `AnswerResult`.


## How the code fits together

`EasyRAG.aquery()` gives you a `QueryResult`, not a finished answer. That matters because the evidence is still visible at this boundary. `select_answer_citations()` trims the raw citation list to something a prompt can actually carry. `build_context_block()` and `build_generation_prompt()` then turn that smaller evidence set into plain text artifacts you can inspect. `generate_answer()` is a thin wrapper around the same steps, plus `synthesize_answer()`. The final `AnswerResult.metadata` keeps both the answer settings and the retrieval metadata that produced the evidence.

## What this cell is proving

We only need the public retrieval and generation APIs, plus a few notebook helpers. The repo-root helper replaces the long bootstrap block that used to show up in every notebook.

In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "_utils.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

_notebook_utils = importlib.import_module("notebooks._utils")
_rag = importlib.import_module("easyrag.rag")
_packing = importlib.import_module("easyrag.rag.generation.packing")
_pipeline = importlib.import_module("easyrag.rag.generation.pipeline")
_prompting = importlib.import_module("easyrag.rag.generation.prompting")
_selection = importlib.import_module("easyrag.rag.generation.selection")
_async_utils = importlib.import_module("easyrag.support.async_utils")

ensure_repo_root_on_path = _notebook_utils.ensure_repo_root_on_path
_print_json = _notebook_utils.print_json
can_use_openai_compatible_models = _notebook_utils.provider_ready
skip_message = _notebook_utils.skip_message
_stub_embedding = _notebook_utils.stub_embedding
_stub_query_model = _notebook_utils.stub_query_model
AnswerParam = _rag.AnswerParam
EasyRAG = _rag.EasyRAG
QueryParam = _rag.QueryParam
build_context_block = _packing.build_context_block
generate_answer = _pipeline.generate_answer
build_generation_prompt = _prompting.build_generation_prompt
select_answer_citations = _selection.select_answer_citations
run_sync = _async_utils.run_sync

REPO_ROOT = ensure_repo_root_on_path()


def flow_stub_answer_model(prompt: str, **kwargs) -> str:
    return (
        "The request moves from Gateway to Policy Engine, then Billing Service "
        "writes invoice state to Ledger Store."
    )

This cell should stay quiet. It sets up the runtime and defines one small deterministic answer model so the final synthesis path stays readable.

## What this cell is proving

The tiny corpus is shaped like a process diagram on purpose. If the citation order is obvious to a human, it becomes much easier to see whether the answer stage is preserving that order or muddying it.

In [ ]:
demo_texts = [
    "# Ingress\nGateway receives billing requests from clients and forwards them to Policy Engine for validation.\n",
    "# Validation\nPolicy Engine checks request rules and passes approved billing work to Billing Service.\n",
    "# Storage\nBilling Service writes invoice state to Ledger Store and emits an event for Reporting Worker.\n",
]
demo_ids = ["doc::ingress", "doc::validation", "doc::storage"]
demo_paths = [
    "architecture/ingress.md",
    "architecture/validation.md",
    "architecture/storage.md",
]

temp_dir = tempfile.TemporaryDirectory()
rag = EasyRAG(
    working_dir=temp_dir.name,
    workspace="generation-overview-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(rag.initialize_storages())
insert_stats = run_sync(rag.ainsert(demo_texts, ids=demo_ids, file_paths=demo_paths))

_print_json(
    {
        "workspace": rag.workspace,
        "paths": demo_paths,
        "insert_stats": insert_stats,
    }
)

## Why this output looks like this

`ainsert()` runs the normal indexing path, so the stats are about derived artifacts, not only raw documents. The exact counts depend on chunking and graph extraction, but the important point is simpler: generation starts from indexed evidence, not from ad hoc strings you hand to a prompt.

## What this cell is proving

`QueryResult` is the real handoff point. Retrieval is finished here, but generation has not started yet. That is why this object still contains raw citations, retrieval mode information, and the query-preparation metadata that led to those citations.

In [ ]:
question = "How does a billing request reach the ledger store?"
query_param = QueryParam(
    mode="hybrid", chunk_top_k=4, rewrite_enabled=True, mqe_enabled=True
)
query_result = run_sync(rag.aquery(question, query_param))

_print_json(
    {
        "mode": query_result.mode,
        "citation_count": len(query_result.citations),
        "metadata": {
            "rewritten_query": query_result.metadata["rewritten_query"],
            "expanded_queries": query_result.metadata["expanded_queries"],
            "retrieval_queries": query_result.metadata["retrieval_queries"],
            "candidate_counts": query_result.metadata["candidate_counts"],
            "vector_backend": query_result.metadata["vector_backend"],
        },
    }
)

for index, citation in enumerate(query_result.citations, start=1):
    print(f"[{index}] {citation['title']} | {citation['location']}")
    print(citation["snippet"])
    print()

## Why this output looks like this

Because rewrite and MQE are enabled, the metadata keeps the original question, the rewritten query, and the expanded variants. The citation list is still raw retrieval output. It may contain more evidence than the answer layer will keep, and it is still ordered by retrieval-side ranking rather than by prompt budget.

## What this cell is proving

The answer stage works on a smaller evidence set than retrieval does. We will call the same library helpers the runtime uses, inspect their outputs, and then run `generate_answer()` so the wrapper is not a black box.

In [ ]:
answer_param = AnswerParam(
    max_citations=2,
    max_context_chars=190,
    style="citation_aware",
    require_citations=True,
    allow_abstain=True,
)
selected_citations = select_answer_citations(
    query_result.citations,
    max_items=answer_param.max_citations,
    max_chars=answer_param.max_context_chars,
)
context_block = build_context_block(selected_citations)
prompt = build_generation_prompt(question, context_block, answer_param)
answer_result = generate_answer(
    question,
    query_result,
    answer_param=answer_param,
    answer_model_func=flow_stub_answer_model,
)

_print_json(
    {
        "selected_locations": [citation["location"] for citation in selected_citations],
        "context_block": context_block,
        "prompt_preview": prompt.splitlines()[:12],
        "answer": answer_result.answer,
        "answer_metadata": {
            "raw_citation_count": answer_result.metadata["raw_citation_count"],
            "selected_citation_count": answer_result.metadata[
                "selected_citation_count"
            ],
            "context_chars": answer_result.metadata["context_chars"],
            "retrieval_mode": answer_result.metadata["retrieval_mode"],
            "answer_model_used": answer_result.metadata["answer_model_used"],
            "fallback_used": answer_result.metadata["fallback_used"],
        },
    }
)

## Why this output looks like this

The raw citation count is larger than the selected count because `select_answer_citations()` applies an answer-side budget. The prompt preview is built from the selected citations, not from the whole retrieval result. `generate_answer()` runs the same selection, packing, and prompting steps again, then hands the prompt to `synthesize_answer()`. Because we passed a working stub answer model, `answer_model_used` is `true` and `fallback_used` is `false`.

## What this cell is proving

`AnswerResult.metadata` keeps the retrieval story alive. If the answer looks wrong, you do not have to guess what the retriever did. The answer object still carries the retrieval metadata that produced its evidence.

In [ ]:
_print_json(
    {
        "selected_vs_raw": {
            "raw_locations": [
                citation["location"] for citation in query_result.citations
            ],
            "selected_locations": [
                citation["location"] for citation in answer_result.selected_citations
            ],
        },
        "retrieval_metadata_in_answer": {
            "rewritten_query": answer_result.metadata["retrieval_metadata"][
                "rewritten_query"
            ],
            "retrieval_queries": answer_result.metadata["retrieval_metadata"][
                "retrieval_queries"
            ],
            "candidate_counts": answer_result.metadata["retrieval_metadata"][
                "candidate_counts"
            ],
        },
    }
)

## What this cell is proving

The provider-backed path should keep the same contract. The answer text may change, but the handoff object is still `QueryResult` and the final wrapper is still `AnswerResult`.

In [ ]:
if not can_use_openai_compatible_models():
    print(skip_message("provider"))
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-generation-overview-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                [
                    "# Retrieval\nGrounded retrieval feeds answer generation.\n",
                    "# Policy\nCitation-aware answers keep the output inspectable.\n",
                ],
                ids=["doc::retrieval", "doc::policy"],
                file_paths=["docs/retrieval.md", "docs/policy.md"],
            )
        )
        provider_answer = run_sync(
            provider_rag.aanswer(
                "How does EasyRAG keep the answer inspectable?",
                QueryParam(mode="hybrid", chunk_top_k=3),
                AnswerParam(max_citations=2, max_context_chars=160),
            )
        )
        _print_json(
            {
                "answer": provider_answer.answer,
                "metadata": {
                    key: provider_answer.metadata[key]
                    for key in (
                        "retrieval_mode",
                        "raw_citation_count",
                        "selected_citation_count",
                        "answer_model_used",
                        "fallback_used",
                    )
                },
            }
        )
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()

## Common mistakes / debugging cues

- If the final answer looks weak, inspect `query_result.citations` before you touch the prompt. The bug may still be retrieval.
- Do not confuse raw citations with selected citations. The answer stage almost always carries fewer items than retrieval returned.
- If the answer text changes between local and provider-backed runs, compare `AnswerResult.metadata` first. The wrapper contract should still look familiar even when the model output does not.

In [ ]:
run_sync(rag.finalize_storages())
temp_dir.cleanup()
print("Cleaned up the query-result-to-answer workspace.")

## Takeaway

The generation stage is readable once you keep the objects separate: `QueryResult` holds the raw evidence, the selection and packing helpers build prompt-ready text, and `AnswerResult` records what survived that narrowing.

Next, open [05_02_evidence_selection_and_topk_for_answering.ipynb](05_02_evidence_selection_and_topk_for_answering.ipynb) if you want to stay with the citation budget step. If you already trust the budget logic, jump to [05_03_context_assembly_and_packing.ipynb](05_03_context_assembly_and_packing.ipynb).